In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
import re
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns

# set display options
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

In [5]:
df = pd.read_csv('../../data/spotify_dataset.csv', on_bad_lines='skip')
#df.columns=["user","artist","track","playlist"]
df = df.drop(columns=df.columns[4:124])


df2 = pd.read_csv('../../data/additional_spotify_dataset.csv')
df2 = df2.drop(columns=df2.columns[39:658])



In [16]:
print(df.describe())
df.info()

print(df2.describe())
df2.info()

#df
#df2

                                 user_id   artistname trackname playlistname
count                            1048575      1046373   1048566      1048501
unique                              1561        68568    386261        18328
top     61baddf7207fea410abdc56e680fa869  Johnny Cash     Intro      Starred
freq                               18901         3458       557        94581
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 4 columns):
 #   Column        Non-Null Count    Dtype 
---  ------        --------------    ----- 
 0   user_id       1048575 non-null  object
 1   artistname    1046373 non-null  object
 2   trackname     1048566 non-null  object
 3   playlistname  1048501 non-null  object
dtypes: object(4)
memory usage: 32.0+ MB
                                   Artist(s)  \
count                                 551479   
unique                                127368   
top     Victor J Sefo,Lisi,Mwayz,Sefos.Beats   
freq  

df:
1,048,575 entries
4 columns

df2:
551,479 entries
39 columns


In [17]:

def normalize(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9 ]', ' ', text)   # keep alphanumerics
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['artist_norm'] = df['artistname'].apply(normalize)
df['track_norm']  = df['trackname'].apply(normalize)

df2['artist_norm'] = df2['Artist(s)'].apply(normalize)
df2['track_norm']  = df2['song'].apply(normalize)

merged_exact = pd.merge(
    df, df2,
    on=['artist_norm','track_norm'],
    how='inner'
)


In [ ]:
merged_exact.info()

df_count = 1048575
df2_count = 551479
merged_count = 389510

pct_df = merged_count / df_count * 100
pct_df2 = merged_count / df2_count * 100

print(f"Percent of df merged: {pct_df:.1f}%")
#print(f"Percent of df2 merged: {pct_df2:.1f}%")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 389510 entries, 0 to 389509
Data columns (total 45 columns):
 #   Column                          Non-Null Count   Dtype 
---  ------                          --------------   ----- 
 0   user_id                         389510 non-null  object
 1   artistname                      389510 non-null  object
 2   trackname                       389510 non-null  object
 3   playlistname                    389510 non-null  object
 4   artist_norm                     389510 non-null  object
 5   track_norm                      389510 non-null  object
 6   Artist(s)                       389510 non-null  object
 7   song                            389510 non-null  object
 8   text                            389510 non-null  object
 9   Length                          389510 non-null  object
 10  emotion                         389510 non-null  object
 11  Genre                           389510 non-null  object
 12  Album                         

In [24]:
merged_exact

,user_id,artistname,trackname,playlistname,artist_norm,track_norm,Artist(s),song,text,Length,emotion,Genre,Album,Release Date,Key,Tempo,Loudness (db),Time signature,Explicit,Popularity,Energy,Danceability,Positiveness,Speechiness,Liveness,Acousticness,Instrumentalness,Good for Party,Good for Work/Study,Good for Relaxation/Meditation,Good for Exercise,Good for Running,Good for Yoga/Stretching,Good for Driving,Good for Social Gatherings,Good for Morning Routine,Similar Artist 1,Similar Song 1,Similarity Score 1,Similar Artist 2,Similar Song 2,Similarity Score 2,Similar Artist 3,Similar Song 3,Similarity Score 3
0,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,(The Angels Wanna Wear My) Red Shoes,HARD ROCK 2010,elvis costello,the angels wanna wear my red shoes,Elvis Costello,The Angels Wanna Wear My Red Shoes,"[Chorus] Oh, I used to be disgusted And now, I...",2:47,sadness,"folk,country,new wave",My Aim Is True,22nd July 1977,E Maj,135,-10db,4-Apr,No,38,64,58,90,5,23,6,0,0,0,0,1,0,0,0,0,0,The Association,Along Comes Mary,0.983201,The Rolling Stones,Flight 505,0.981561,Hurriganes,My Only One,0.979894
1,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello & The Attractions,Accidents Will Happen,HARD ROCK 2010,elvis costello the attractions,accidents will happen,Elvis Costello & The Attractions,Accidents Will Happen,"[Verse 1] Oh, I just don't know where to begin...",3:01,sadness,"folk,country,new wave",Armed Forces (Super Deluxe Edition),5th January 1979,C Maj,120,-11.12db,4-Apr,No,37,60,61,74,3,28,4,0,0,0,0,0,0,0,0,0,1,Adam Green,Friends of Mine,0.981811,MAY-A,Time I Love To Waste,0.979441,KOPPS,Dumb,0.975395
2,9cc0cfd4d7d7885102480dd99e7a90d6,Elvis Costello,Alison,HARD ROCK 2010,elvis costello,alison,Elvis Costello,Alison,"[Verse 1] Oh, it's so funny to be seeing you a...",3:24,surprise,"folk,country,new wave",My Aim Is True,22nd July 1977,C# min,177,-10.79db,4-Apr,No,51,32,56,38,4,11,74,0,0,0,0,0,0,0,0,0,0,Alison Krauss & Union Station,When You Say Nothing At All,0.995125,Restless Heart,Ill Still Be Loving You,0.994933,Cavetown,888,0.992878
3,9cc0cfd4d7d7885102480dd99e7a90d6,Paul McCartney,Dance Tonight,HARD ROCK 2010,paul mccartney,dance tonight,Paul McCartney,Dance Tonight,[Refrain] Everybody gonna dance tonight Everyb...,2:54,joy,"pop rock,folk,pop",Memory Almost Full,4th June 2007,F Maj,171,-2.54db,4-Apr,No,35,89,53,93,4,16,8,1,0,0,0,1,1,0,0,0,0,Elemeno P,Chloe Edit,0.994229,Elemeno P,Seventeen,0.993897,Elemeno P,Verona,0.993897
4,9cc0cfd4d7d7885102480dd99e7a90d6,Crowded House,Don't Dream It's Over,HARD ROCK 2010,crowded house,don t dream it s over,Crowded House,Don't Dream It's Over,"There is freedom within, there is freedom with...",3:58,joy,"rock,pop,alternative rock",Crowded House (Deluxe),1st August 1986,G# Maj,82,-6.99db,4-Apr,No,71,75,45,36,4,8,1,0,0,0,0,0,0,0,0,0,0,Crowded House,Dont Dream Its Over,0.999674,The Neighbourhood,A Little Death,0.983522,"Egzod,Maestro Chives,Neoni",The Revolution,0.979075
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
389505,576820c2da503406e16ae9782b3c0e4c,Susto,Dream Girl,Current Office Playlist - regularly updated,susto,dream girl,Susto,Dream Girl,Lately I've been having the strangest dreams I...,3:40,fear,hip hop,Susto,1st April 2014,A Maj,140,-7.39db,4-Apr,No,16,58,60,32,3,11,55,0,0,0,0,0,0,0,0,0,0,"Amy Grant,James Taylor",Here,0.989282,Vonda Shepard,This Is Crazy Now,0.988307,Faith Richards,Blue,0.98816
389506,576820c2da503406e16ae9782b3c0e4c,Billy Joe Shaver,Live Forever,Current Office Playlist - regularly updated,billy joe shaver,live forever,Billy Joe Shaver,Live Forever,"We rollin'? One, two, three I'm gonna live fo...",2:49,sadness,country,The Essential Billy Joe Shaver,2nd October 2015,G Maj,105,-15.19db,4-Apr,No,20,45,66,89,3,33,41,0,0,1,0,0,0,0,0,0,0,Panopticon,Which Side Are You On?,0.97753,Roy Clark,I Never Picked Cotton,0.972329,Tom Jones,Un

merged: 
389,510 entries
45 columns

In [2]:
from rapidfuzz import process, fuzz

matches = []

for artist in df['artist_norm'].unique():
    left = df[df['artist_norm'] == artist]
    right = df2[df2['artist_norm'] == artist]
    
    for i, track in left['track_norm'].items():
        match = process.extractOne(
            track, right['track_norm'],
            scorer=fuzz.token_sort_ratio
        )
        if match and match[1] > 70:  # similarity threshold
            # find the index of the matched track in df2
            j = right[right['track_norm'] == match[0]].index[0]
            matches.append((i, j, match[1]))  # store indices + score

# Convert matches to DataFrame
matches_df = pd.DataFrame(matches, columns=['df_index','df2_index','score'])

# Join back to full rows
fuzzy_merged = matches_df.merge(df, left_on='df_index', right_index=True)\
                         .merge(df2, left_on='df2_index', right_index=True,
                                suffixes=('_df','_df2'))

NameError: name 'df' is not defined

In [1]:
#matches_df.sort_values(by='user_id', ascending=False)
fuzzy_merged.sort_values(by='score') 

NameError: name 'fuzzy_merged' is not defined

merged: 
389,510 entries
45 columns